In [1]:
import os
os.environ['USER_AGENT'] = 'myagent'

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter

In [4]:
import pandas as pd
data = pd.read_csv("..\data\ecommerceDataset.csv")

In [5]:
data.iloc[:,1]

0        SAF 'Floral' Framed Painting (Wood, 30 inch x ...
1        SAF 'UV Textured Modern Art Print Framed' Pain...
2        SAF Flower Print Framed Painting (Synthetic, 1...
3        Incredible Gifts India Wooden Happy Birthday U...
4        Pitaara Box Romantic Venice Canvas Painting 6m...
                               ...                        
50419    Strontium MicroSD Class 10 8GB Memory Card (Bl...
50420    CrossBeats Wave Waterproof Bluetooth Wireless ...
50421    Karbonn Titanium Wind W4 (White) Karbonn Titan...
50422    Samsung Guru FM Plus (SM-B110E/D, Black) Colou...
50423                     Micromax Canvas Win W121 (White)
Name: Paper Plane Design Framed Wall Hanging Motivational Office Decor Art Prints (8.7 X 8.7 inch) - Set of 4 Painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch. This painting is ready to hang

## Making vector data from text with embedding models

In [6]:
text = data.iloc[100,1]
text

'AsianHobbyCrafts Wooden Embroidery Hoop Ring Frame (3 Pieces) Style Name:Assorted A   Asian Hobby Crafts embroidery collection comprises of embroidery frames (in various sizes), cross stitch fabric, embroidery tools, embroidery wool. This embroidery hoop frame is made of well finished wood with a easy-to-adjust screw mounted on the frame to tighten the fabric. Cross stitch art is a phenomenal art form which involves intricate stitching techniques to form beautiful designs on fabric.'

In [7]:
# lets choose a hf embedding model
from sentence_transformers import SentenceTransformer
model_name = "sentence-transformers/all-mpnet-base-v2"
model = SentenceTransformer(model_name)

embeddings1 = model.encode(text)

In [13]:
# embeddings1

In [9]:
# we can do the same thing with langchain's HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings # other embeddings available
from langchain_community.vectorstores import Chroma

model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

# create embeddings
model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
embeddings2 = model.embed_query(text)

In [11]:
# embeddings2

In [13]:
list(embeddings1) == list(embeddings2)

True

## Vector database from a list of strings

In [14]:
# only taking 100 rows
texts = list(data.iloc[:100,1])
# print(texts[:10])

In [18]:
#storing the data in Vector Store
from langchain_chroma import Chroma
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

# create embeddings
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
persist_directory = os.path.join("..", "data", "ecommerce_vectordb")

# database
vectordb = Chroma.from_texts(texts=texts, embedding=embedding_model, persist_directory=persist_directory)
# vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
# vectordb = Chroma.from_texts(texts=texts, embedding=embedding_model) # takes a list o../.f strings

In [19]:
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)

In [21]:
persist_directory

'..\\data\\ecommerce_vectordb'

## From a text document

In [16]:
# raw_documents = TextLoader('data_example.txt').load()
# text_splitter = CharacterTextSplitter(separator = '.', chunk_size=10, chunk_overlap=0)
# data = text_splitter.split_documents(raw_documents)

Created a chunk of size 12, which is longer than the specified 10
Created a chunk of size 27, which is longer than the specified 10


In [22]:
# data

In [18]:
# #storing the data in Vector Store
# model_name = "sentence-transformers/all-mpnet-base-v2"
# model_kwargs = {"device": "cuda"}
#
# # create embeddings
# embedding = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
#
# # database
# vector_database = Chroma.from_documents(documents=data, embedding=embedding) # takes a list of text documents